# 🇮🇳 TamilTech-QA — Live Demo (Both Models)

Compare **base Llama-3.1-8B** vs. our **QLoRA fine-tuned** model on Tanglish (Tamil-English) technical questions.

**Free on Colab T4 GPU.** Setup takes ~5 min, then each query answers in ~5 sec.

## Before you run
1. **Runtime → Change runtime type → T4 GPU** (top menu)
2. **Runtime → Run all** (or run cells one-by-one with Shift+Enter)
3. When prompted, paste your **HuggingFace token** (you need Llama-3.1 access). Get one at https://huggingface.co/settings/tokens
4. After ~5 min, a public Gradio URL appears at the bottom — open it to chat with both models

## Links
- Model: [dheepakkaran/TamilTech-QA-Llama3.1-8B-QLoRA](https://huggingface.co/dheepakkaran/TamilTech-QA-Llama3.1-8B-QLoRA)
- Dataset: [dheepakkaran/TamilTech-QA](https://huggingface.co/datasets/dheepakkaran/TamilTech-QA)
- Space: [TamilTech-QA-Demo](https://huggingface.co/spaces/dheepakkaran/TamilTech-QA-Demo)

## Step 1 — Install dependencies (~2 min)

In [ ]:
!pip install -q -U transformers>=4.46.0 peft>=0.13.0 bitsandbytes>=0.44.0 gradio>=4.44.0 accelerate>=1.1.0

## Step 2 — Login to HuggingFace (paste your HF token when prompted)

Need access to Llama-3.1-8B-Instruct (gated). Token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()  # paste your HF token in the prompt below

## Step 3 — Load BOTH models in 4-bit (~3-5 min first time)

Downloads ~16 GB Llama base model + 27 MB adapter. Both fit in T4's 16 GB VRAM thanks to 4-bit quantization.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = 'meta-llama/Llama-3.1-8B-Instruct'
ADAPTER = 'dheepakkaran/TamilTech-QA-Llama3.1-8B-QLoRA'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base Llama-3.1-8B (this takes 3-5 min first time)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16,
)
base_model.eval()
print('Base model loaded.')

print('Loading fine-tuned LoRA adapter...')
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16,
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER)
ft_model.eval()
print('Fine-tuned model loaded.')
print()
print('Both models ready! Continue to next cell.')

## Step 4 — Define inference + Tanglish metrics

In [ ]:
import re

TAMIL_HINTS = set('enna na neenga indha intha sollunga puriyutha theriyuma epdi paaru naa ennoda apdi maari konjam vendam summa thaan irukku appram appo aana naan ninga puriyala patha sonna sollra panrathu panra panren panni pannunga vanthu vandhu varuthu sema semma yenna aprum aanaa adhu idhu antha athu maathiri mathiri yappadi ennamo dhan'.split())

TECH = ['python','function','array','loop','class','gradient','model','algorithm','tensor','network','neural','cnn','rnn','lstm','transformer','decorator','pointer','memory','kernel','process','thread','rest','api','http','sql','database','stack','queue','tree','graph','hash','binary','compile','debug','exception','error','javascript','react','node','backend','frontend','docker','aws','cloud','linux','optimization','minimize','derivative']

CONN = ['enna na','apdi patha','sollunga','puriyutha','theriyuma','indha maari','konjam','vendam','aana','aprm','appram','appo','ipdi','epdi','paaru']

def ratio(s):
    toks = re.findall(r"[a-zA-Z][a-zA-Z\-']*", (s or '').lower())
    return 0.0 if not toks else sum(1 for t in toks if t in TAMIL_HINTS) / len(toks)

def tech_count(s):
    t = (s or '').lower()
    return sum(1 for k in TECH if re.search(r'\b' + re.escape(k) + r'\b', t))

def conn_count(s):
    t = (s or '').lower()
    return sum(1 for c in CONN if c in t)

def generate(model, prompt, max_new=200):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def compare(question, max_new=200):
    if not question or not question.strip():
        return 'Please enter a question.', '', '', ''
    prompt = (
        '<|im_start|>system\n'
        'You are a helpful Tanglish technical assistant. Answer in Tamil-English mixed style.<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    base_text = generate(base_model, prompt, max_new)
    ft_text = generate(ft_model, prompt, max_new)
    base_m = (
        f'Tanglish: {ratio(base_text):.1%} | '
        f'Tech: {tech_count(base_text)} | '
        f'Connectors: {conn_count(base_text)} | '
        f'Words: {len(base_text.split())}'
    )
    ft_m = (
        f'Tanglish: {ratio(ft_text):.1%} | '
        f'Tech: {tech_count(ft_text)} | '
        f'Connectors: {conn_count(ft_text)} | '
        f'Words: {len(ft_text.split())}'
    )
    return base_text, ft_text, base_m, ft_m

print('Inference functions ready.')

## Step 5 — Launch Gradio UI

After this cell runs, **a public gradio.live URL appears in the output**. Open it in any browser to use the live demo.

The URL is shareable for ~72 hours.

In [ ]:
import gradio as gr

HEADER = '''
# 🇮🇳 TamilTech-QA — Live Demo

**Compare base Llama-3.1-8B vs. our QLoRA fine-tuned model** on Tanglish technical questions.

- Each answer generates in ~5 seconds on T4
- Both models running in 4-bit quantization (fits in 16 GB VRAM)
- Live Tanglish metrics shown below each output
'''

EXAMPLES = [
    ['indha gradient descent enna na konjam explain pannunga'],
    ['python decorator epdi work pannuthu sollunga'],
    ['linked list and array difference enna'],
    ['modulation enna, simple-a sollunga'],
    ['REST API enna mathiri work pannuthu intha concept-a clear-a sollunga'],
    ['binary search algorithm epdi implement pannrathu'],
]

with gr.Blocks(title='TamilTech-QA Live', theme=gr.themes.Soft()) as demo:
    gr.Markdown(HEADER)

    with gr.Row():
        question = gr.Textbox(
            label='Ask a Tanglish technical question',
            placeholder='indha gradient descent enna na konjam explain pannunga',
            lines=2,
        )
    with gr.Row():
        with gr.Column(scale=3):
            run_btn = gr.Button('🚀 Compare both models', variant='primary')
        with gr.Column(scale=1):
            max_tokens = gr.Slider(50, 400, value=200, step=10, label='Max new tokens')

    with gr.Row():
        with gr.Column():
            gr.Markdown('### 🤖 Base Llama-3.1-8B (zero-shot)')
            base_out = gr.Textbox(label='Output', lines=10, interactive=False)
            base_m = gr.Textbox(label='Live Tanglish metrics', lines=2, interactive=False)
        with gr.Column():
            gr.Markdown('### ⭐ TamilTech-QA (QLoRA fine-tuned)')
            ft_out = gr.Textbox(label='Output', lines=10, interactive=False)
            ft_m = gr.Textbox(label='Live Tanglish metrics', lines=2, interactive=False)

    gr.Examples(examples=EXAMPLES, inputs=question, label='Click any example to try it:')

    gr.Markdown('''
---
### About
Built with QLoRA fine-tuning on 4,415 Tanglish QA samples. Model achieves 78% perplexity reduction vs. base.  
Citation: TamilTech-QA (2026) by Dheepak Karan E S.
''')

    run_btn.click(
        compare,
        inputs=[question, max_tokens],
        outputs=[base_out, ft_out, base_m, ft_m],
    )

demo.launch(share=True, debug=False)

## What to do next

1. **Open the gradio.live URL** that just printed above
2. Type any Tanglish technical question
3. Watch both models answer side-by-side
4. The URL is shareable with anyone for ~72 hours
5. Keep this Colab tab open to keep the demo alive

**To restart fresh:** Runtime → Restart and run all